#### Modelo extração de texto Marckdown e JSON

##### obs.: Para usa essa lib no windows é preciso ativar o modo desenvolvedor

In [2]:
from docling.document_converter import DocumentConverter
from tqdm import tqdm

source = r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\pdf\citros.pdf"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)
print(result.document.export_to_markdown())


## 3.4. Citros

## (Gênero Citrus )

José Antonio Quaggio ( 1 ) Dirceu Mattos Jr. ( 1 ) Rodrigo Marcelli Boaretto ( 1 ) Fernando Cesar Bachiega Zambrosi ( 1 ) Heitor Cantarella ( 1 )

## 1. Considerações gerais

A citricultura  contribui  com  o  maior  volume  da  produção brasileira  de  frutas  (cerca  de  55%  do  volume  das  oito principais frutíferas), obtido  com  alta  produtividade, resultado  da  adoção  de  tecnologias  como  adensamento  de  plantio, adubação, irrigação e fertirrigação. Os citros compreendem um grande grupo de plantas do gênero Citrus spp. (laranjas, tangerinas, mexericas, limões, limas ácidas como o Tahiti e o Galego, e doces como a Lima-da-pérsia,  pomelo,  cidra,  laranja-azeda  e  toranjas)  e  outros  gêneros correlatos ( Fortunella e Poncirus ) ou híbridos da família Rutaceae. Devido à  importância  da  citricultura  para  o  agronegócio  paulista  e  o  volume de  informações  geradas  pela  pesquisa,  as  recomendações  de  manejo de  calagem  e  a

In [ ]:
from transformers import AutoTokenizer

from docling.chunking import HybridChunker

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 64  # set to a small number for illustrative purposes

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

chunker = HybridChunker(
    tokenizer=tokenizer,  # instance or model name, defaults to "sentence-transformers/all-MiniLM-L6-v2"
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer`
    merge_peers=True,  # optional, defaults to True
)
chunk_iter = chunker.chunk(dl_doc=doc)
chunks = list(chunk_iter)

In [ ]:
from transformers import AutoTokenizer
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

conv_res = DocumentConverter().convert(r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\pdf\citros.pdf")
doc = conv_res.document

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 300  # set to a small number for illustrative purposes

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

chunker = HybridChunker(
    tokenizer=tokenizer,  # instance or model name, defaults to "sentence-transformers/all-MiniLM-L6-v2"
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer`
    merge_peers=True,  # optional, defaults to True
)
chunk_iter = chunker.chunk(dl_doc=doc)
chunks = list(chunk_iter)

for i, chunk in enumerate(chunks):
    print(f"=== {i} ===")
    txt_tokens = len(tokenizer.tokenize(chunk.text, max_length=None))
    print(f"chunk.text ({txt_tokens} tokens):\n{repr(chunk.text)}")

    ser_txt = chunker.serialize(chunk=chunk)
    ser_tokens = len(tokenizer.tokenize(ser_txt, max_length=None))
    print(f"chunker.serialize(chunk) ({ser_tokens} tokens):\n{repr(ser_txt)}")

    print()

In [2]:
import os
import json
import pandas as pd
from pdf2image import convert_from_path
from PyPDF2 import PdfReader
import pytesseract
import glob

# Configuração do Tesseract OCR (adicione o caminho do executável, se necessário)
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

def is_readable_pdf(file_path):
    """Verifica se o PDF é legível ou contém imagens que necessitam OCR."""
    try:
        reader = PdfReader(file_path)
        first_page = reader.pages[0]
        text = first_page.extract_text()
        return text is not None and bool(text.strip())  # Retorna True se encontrar texto legível
    except Exception as e:
        print(f"Erro ao verificar o PDF {file_path}: {e}")
        return False

def extract_text_from_pdf(file_path):
    """Extrai texto diretamente de um PDF legível."""
    text = []
    try:
        reader = PdfReader(file_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
    except Exception as e:
        print(f"Erro ao extrair texto do PDF {file_path}: {e}")
    return "\n".join(text)

def perform_ocr_on_pdf(file_path):
    """Realiza OCR em um PDF baseado em imagens."""
    ocr_text = []
    try:
        images = convert_from_path(file_path)
        for image in images:
            ocr_text.append(pytesseract.image_to_string(image))
    except Exception as e:
        print(f"Erro ao realizar OCR no PDF {file_path}: {e}")
    return "\n".join(ocr_text)

def save_to_files(text, output_prefix):
    """Salva o texto extraído em formatos JSON, Markdown, CSV."""
    # JSON
    json_path = f"{output_prefix}.json"
    with open(json_path, "w", encoding="utf-8") as json_file:
        json.dump({"text": text}, json_file, ensure_ascii=False, indent=4)
    
    # Markdown
    markdown_path = f"{output_prefix}.md"
    with open(markdown_path, "w", encoding="utf-8") as md_file:
        md_file.write(text)
    
    # CSV
    csv_path = f"{output_prefix}.csv"
    df = pd.DataFrame([{"content": text}])
    df.to_csv(csv_path, index=False, encoding="utf-8")
    
    print(f"Resultados salvos como JSON: {json_path}, Markdown: {markdown_path}, CSV: {csv_path}")

def process_documents(file_paths):
    """Processa uma lista de documentos PDF e salva os resultados."""
    for file_path in file_paths:
        print(f"Processando: {file_path}")
        if is_readable_pdf(file_path):
            print("PDF legível detectado.")
            extracted_text = extract_text_from_pdf(file_path)
        else:
            print("PDF baseado em imagens detectado. Iniciando OCR.")
            extracted_text = perform_ocr_on_pdf(file_path)
        
        # Definir o prefixo de saída
        output_prefix = os.path.splitext(os.path.basename(file_path))[0]
        save_to_files(extracted_text, output_prefix)

tipos = [
    # "decisão",
    # "inquérito policial",
    # "laudo iml",
    # "laudo médico",
    # "laudo médico (digitalizado)",
    # "laudo pericial",
    # "laudo pericial (digitalizado)",
    #"relatório de investigações", Retirei por que os documentos não são relevantes. 
    #"relatório final",
    "relatório final digitalizado",
    #"sentença"
]

# Iterando por cada tipo de documento
dados = []
for pasta in tipos:
    # Pegando os primeiros 5 arquivos PDF de cada tipo de pasta
    source = glob.glob(f"C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/{pasta}/*.pdf")
    # Processar os PDFs
    dados+=process_documents(source)

Processando: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/relatório final digitalizado\0000079-22.2020.8.26.0052_relatório_final_digitalizado_unificado.pdf
PDF legível detectado.
Resultados salvos como JSON: 0000079-22.2020.8.26.0052_relatório_final_digitalizado_unificado.json, Markdown: 0000079-22.2020.8.26.0052_relatório_final_digitalizado_unificado.md, CSV: 0000079-22.2020.8.26.0052_relatório_final_digitalizado_unificado.csv
Processando: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/relatório final digitalizado\0000105-20.2020.8.26.0052_relatório_final_digitalizado_unificado.pdf
PDF legível detectado.
Resultados salvos como JSON: 0000105-20.2020.8.26.0052_relatório_final_digitalizado_unificado.json, Markdown: 0000105-20.2020.8.26.0052_relatório_final_digitalizado_unificado.md, CSV: 0000105-20.2020.8.26.0052_relatório_final_digitalizado_unificado.csv
Processando: C

TypeError: 'NoneType' object is not iterable

In [ ]:
# salva uma página

from PIL import Image
import fitz
import os

# Caminho do PDF
source = r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\pdf\citros.pdf"

# Abrir o documento PDF
pdf_document = fitz.open(source)

# Criar diretório para salvar as imagens, se não existir
output_dir = r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base"
os.makedirs(output_dir, exist_ok=True)

# Salvar cada página como uma imagem em preto e branco com 300 DPI
for i in range(len(pdf_document)):
    page = pdf_document.load_page(i)
    pix = page.get_pixmap(dpi=300)
    image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    bw_image = image.convert("L")  # Converter para preto e branco
    image_path = os.path.join(output_dir, f"page_{i + 1}.png")
    bw_image.save(image_path, dpi=(300, 300))
    print(f"Página {i + 1} salva como {image_path}")

In [ ]:
## salva com todas as páginas
from PIL import Image
import fitz
import os

# Caminho do PDF
source = r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\pdf\citros.pdf"

# Abrir o documento PDF
pdf_document = fitz.open(source)

# Criar diretório para salvar as imagens, se não existir
output_dir = r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\1 - criação_base"
os.makedirs(output_dir, exist_ok=True)

# Lista para armazenar as imagens das páginas
images = []

# Salvar cada página como uma imagem em preto e branco com 300 DPI
for i in range(len(pdf_document)):
    page = pdf_document.load_page(i)
    pix = page.get_pixmap(dpi=300)
    image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    bw_image = image.convert("L")  # Converter para preto e branco
    images.append(bw_image)

# Combinar todas as imagens verticalmente
total_height = sum(image.height for image in images)
max_width = max(image.width for image in images)
combined_image = Image.new('L', (max_width, total_height))

y_offset = 0
for image in images:
    combined_image.paste(image, (0, y_offset))
    y_offset += image.height

# Salvar a imagem combinada
combined_image_path = os.path.join(output_dir, "combined_pages.png")
combined_image.save(combined_image_path, dpi=(300, 300))
print(f"Todas as páginas salvas como {combined_image_path}")

In [ ]:
import os
from pdf2image import convert_from_path
import pytesseract

# Configuração do Tesseract OCR (adicione o caminho do executável, se necessário)
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

def perform_ocr_on_pdf(file_path):
    """Realiza OCR em um PDF baseado em imagens e imprime o texto extraído."""
    ocr_text = []
    try:
        images = convert_from_path(file_path)
        for image in images:
            text = pytesseract.image_to_string(image)
            ocr_text.append(text)
            print(text)  # Imprime o texto extraído de cada página
    except Exception as e:
        print(f"Erro ao realizar OCR no PDF {file_path}: {e}")
    return "\n".join(ocr_text)

# Caminho do PDF de exemplo
pdf_path = r"C:\Estudos e Projetos\(2024) LLM Agrônomo Virtual\1 - criação_base\page_1.pdf"

# Realizar OCR no PDF e imprimir o texto extraído
perform_ocr_on_pdf(pdf_path)